# Etapa 3 — Análise Exploratória e Consultas SQL
## Classificação de Sentenças Regulatórias — CODE-ACCORD

**Disciplina:** Ciência de Dados: Análise de Dados Aplicada  
**Dataset:** CODE-ACCORD (Building Regulations — England & Finland)  
**Referência:** Hettiarachchi et al., *Scientific Data*, 2025

---

### Estrutura deste notebook

| Seção | Conteúdo |
|-------|----------|
| 0 | Setup e carregamento das bases |
| 1 | Análise estrutural das 3 bases |
| 2 | Preparação Tidy + exportação Parquet |
| 3 | Consultas SQL analíticas (DuckDB) |
| 4 | Análise univariada |
| 5 | Análise bivariada — marcadores linguísticos |
| 6 | IAA entre bases (Cohen's Kappa) |
| 7 | Análise multivariada — PCA e clustering |
| 8 | Testes de hipóteses formais |
| 9 | Síntese e insights para modelagem |


## Seção 0 — Setup e Carregamento

Instalação e importação de todas as bibliotecas necessárias. Execute esta célula primeiro.

In [ ]:
# ── Instalação (execute apenas uma vez) ──────────────────────────────────────
# !pip install duckdb pyarrow scikit-learn scipy matplotlib seaborn -q


In [ ]:
# ── Importações ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import re
import warnings
from collections import Counter
from scipy import stats
from scipy.stats import mannwhitneyu, fisher_exact, chi2_contingency
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Paleta de cores consistente em todo o notebook
C_RULE    = '#2E86AB'   # azul  → regra (self-contained)
C_NORULE  = '#E84855'   # vermelho → não-regra (others)
C_BASE1   = '#2E86AB'   # ACCORD Official
C_BASE2   = '#E84855'   # Nossa base
C_BASE3   = '#F6AE2D'   # ALL.csv baseline
C_BG      = '#F8F9FA'
C_GRID    = '#DEE2E6'

plt.rcParams.update({
    'figure.facecolor': C_BG,
    'axes.facecolor':   C_BG,
    'axes.grid':        True,
    'grid.color':       C_GRID,
    'grid.linewidth':   0.8,
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('✅ Bibliotecas carregadas com sucesso.')
print(f'   duckdb   : {duckdb.__version__}')
print(f'   pyarrow  : {pa.__version__}')
print(f'   pandas   : {pd.__version__}')


### 0.1 — Caminhos dos arquivos

> **Ajuste os caminhos abaixo conforme seu ambiente (Colab, local, etc.)**

In [ ]:
# ── CONFIGURE OS CAMINHOS AQUI ───────────────────────────────────────────────
PATH_ACCORD_OFFICIAL = 'accord_official_clean.csv'      # Base ACCORD tratada
PATH_NOSSA_BASE      = 'classification_dataset_cleaned.csv'  # Nossa base
PATH_ALL_CSV         = 'all.csv'                        # Baseline 862 sentenças
OUTPUT_PARQUET       = 'dataset_tidy.parquet'           # Saída Parquet (Etapa 2)
# ─────────────────────────────────────────────────────────────────────────────


### 0.2 — Carregamento e padronização

In [ ]:
# ── BASE 1: ACCORD Official ──────────────────────────────────────────────────
df_official = pd.read_csv(PATH_ACCORD_OFFICIAL)
# Garante colunas padronizadas
df_official = df_official.rename(columns={c: c.strip() for c in df_official.columns})
df_official['base'] = 'ACCORD Official'
df_official['Text'] = df_official['Text'].str.strip()
print(f'BASE 1 — ACCORD Official: {df_official.shape}')
print(df_official['is_rule'].value_counts().to_dict())
print()

# ── BASE 2: Nossa base construída ────────────────────────────────────────────
df_ours = pd.read_csv(PATH_NOSSA_BASE)
df_ours = df_ours.rename(columns={c: c.strip() for c in df_ours.columns})
df_ours['base'] = 'Nossa Base'
df_ours['Text'] = df_ours['Text'].str.strip()
print(f'BASE 2 — Nossa Base: {df_ours.shape}')
print(df_ours['is_rule'].value_counts().to_dict())
print()

# ── BASE 3: ALL.csv — baseline anotado (862 sentenças) ───────────────────────
df_all = pd.read_csv(PATH_ALL_CSV)
df_all = df_all.rename(columns={'content': 'Text'})
df_all['is_rule'] = 1        # Todas são self-contained por definição
df_all['base']    = 'ALL.csv (Baseline)'
df_all['Text']    = df_all['Text'].str.strip()
print(f'BASE 3 — ALL.csv Baseline: {df_all.shape}')
print(df_all['is_rule'].value_counts().to_dict())


---
## Seção 1 — Análise Estrutural das Três Bases

Antes de qualquer análise comparativa, é fundamental entender como cada base foi construída
e quais são suas características estruturais. Isso responde à pergunta: **as três bases são 
comparáveis entre si?**


In [ ]:
# ── Tabela resumo das três bases ─────────────────────────────────────────────
summary = []
for df, name in [(df_official,'ACCORD Official'), (df_ours,'Nossa Base'), (df_all,'ALL.csv')]:
    n_total = len(df)
    n_rules = (df['is_rule'] == 1).sum()
    n_other = (df['is_rule'] == 0).sum()
    ratio   = n_other / n_rules if n_rules > 0 else float('inf')
    summary.append({
        'Base': name,
        'Total': n_total,
        'Regras (1)': n_rules,
        'Não-Regras (0)': n_other,
        '% Regras': f'{n_rules/n_total*100:.1f}%',
        'Razão 0:1': f'1:{ratio:.1f}'
    })

df_summary = pd.DataFrame(summary)
print('╔══ RESUMO DAS TRÊS BASES ══════════════════════════════════════════╗')
print(df_summary.to_string(index=False))
print('╚═══════════════════════════════════════════════════════════════════╝')
print()
print('📌 Interpretação:')
print('   • ALL.csv contém apenas positivos — é o baseline anotado por especialistas.')
print('   • ACCORD Official tem o melhor balanceamento (1:1.7).')
print('   • Nossa Base tem o maior volume total mas maior desbalanceamento (1:2.6).')


In [ ]:
# ── Gráfico: distribuição das classes por base ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribuição de Classes por Base de Dados', fontsize=14, fontweight='bold')

for ax, (df, name, color) in zip(axes, [
    (df_official, 'ACCORD Official', C_BASE1),
    (df_ours,     'Nossa Base',       C_BASE2),
    (df_all,      'ALL.csv',          C_BASE3)
]):
    counts = df['is_rule'].value_counts().sort_index()
    labels = ['Não-Regra\n(Others)', 'Regra\n(Self-contained)'][:len(counts)]
    colors = [C_NORULE, C_RULE][:len(counts)]
    bars   = ax.bar(labels, counts.values, color=colors, width=0.5,
                    edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        pct = val / len(df) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_ylabel('Nº de Sentenças')
    ax.set_ylim(0, max(counts.values) * 1.25)

plt.tight_layout()
plt.savefig('fig1_distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig1_distribuicao_classes.png')


---
## Seção 2 — Preparação Tidy e Exportação Parquet

Seguindo os princípios de **tidy data** (Wickham, 2014):
- Cada variável → uma coluna
- Cada observação → uma linha  
- Cada tipo de unidade observacional → uma tabela

Aqui consolidamos as três bases em um único dataset tidy e extraímos features linguísticas.


In [ ]:
# ── Função de extração de features ──────────────────────────────────────────
DEONTIC      = ['shall','must','should','may','cannot','must not','shall not',
                'is required','are required','is prohibited']
QUANTITATIVE = ['at least','no more than','not exceed','not less than',
                'minimum','maximum','greater than','less than','equal to']
COREF        = ['it ','they ','this ','these ','such ','those ','the above',
                'the following','as specified','as described']
EXT_REF      = ['section','table','figure','appendix','clause','see ','refer to',
                'paragraph','schedule','annex']

def extract_features(text):
    """Extrai features linguísticas de uma sentença."""
    if pd.isna(text):
        return {}
    t  = str(text).strip()
    tl = t.lower()
    tokens = re.sub(r'[^\w\s]', ' ', tl).split()
    return {
        'n_tokens':       len(tokens),
        'n_chars':        len(t),
        'n_deontic':      sum(1 for w in DEONTIC      if w in tl),
        'n_quantitative': sum(1 for w in QUANTITATIVE if w in tl),
        'n_coref':        sum(1 for w in COREF        if w in tl),
        'n_ext_ref':      sum(1 for w in EXT_REF      if w in tl),
        'has_shall':      int('shall'  in tl),
        'has_must':       int('must'   in tl),
        'has_should':     int('should' in tl),
        'has_may':        int('may'    in tl),
        'starts_capital': int(t[0].isupper() if t else False),
        'ends_period':    int(t.endswith('.')),
        'n_commas':       t.count(','),
        'n_semicolons':   t.count(';'),
        'n_numbers':      len(re.findall(r'\b\d+\.?\d*\b', t)),
    }

print('⚙️  Extraindo features de todas as bases...')

# Concatena as três bases com colunas padronizadas
df_combined = pd.concat([
    df_official[['Text','is_rule','base']],
    df_ours    [['Text','is_rule','base']],
    df_all     [['Text','is_rule','base']],
], ignore_index=True)

# Remove duplicatas — mantém primeira ocorrência (ACCORD Official tem prioridade)
df_combined = df_combined.drop_duplicates(subset=['Text'], keep='first')
print(f'Total de sentenças únicas: {len(df_combined)}')

# Extrai features
features = df_combined['Text'].apply(extract_features).apply(pd.Series)
df_tidy   = pd.concat([df_combined.reset_index(drop=True), features], axis=1)

# Tipos corretos
int_cols = [c for c in df_tidy.columns if c not in ['Text','base']]
for c in int_cols:
    df_tidy[c] = pd.to_numeric(df_tidy[c], errors='coerce').fillna(0)

df_tidy['is_rule'] = df_tidy['is_rule'].astype(int)

print(f'Shape tidy: {df_tidy.shape}')
print(f'Colunas: {df_tidy.columns.tolist()}')
df_tidy.head(3)


In [ ]:
# ── Exporta para Parquet ─────────────────────────────────────────────────────
table = pa.Table.from_pandas(df_tidy)
pq.write_table(table, OUTPUT_PARQUET, compression='snappy')
print(f'✅ Dataset exportado para: {OUTPUT_PARQUET}')
print(f'   Linhas: {len(df_tidy):,} | Colunas: {len(df_tidy.columns)} | '
      f'Tamanho: {table.nbytes/1024:.1f} KB (em memória)')
print()
# Verifica a leitura do Parquet
df_check = pd.read_parquet(OUTPUT_PARQUET)
print(f'   Verificação de leitura: {df_check.shape} ✓')


---
## Seção 3 — Consultas SQL Analíticas com DuckDB

DuckDB permite consultas SQL diretamente sobre DataFrames pandas e arquivos Parquet,
sem necessidade de banco de dados externo. Todas as consultas abaixo usam o dataset tidy.


In [ ]:
# ── Registra o dataset no DuckDB ─────────────────────────────────────────────
con = duckdb.connect()
con.register('sentences', df_tidy)
print('✅ Dataset registrado no DuckDB como tabela "sentences".')
print()
con.execute('SELECT COUNT(*) as total, COUNT(DISTINCT base) as bases FROM sentences').df()


### SQL 1 — Distribuição de classes por base de dados

In [ ]:
# ── SQL 1: Distribuição de classes por base ──────────────────────────────────
sql1 = """
SELECT
    base,
    COUNT(*)                                          AS total,
    SUM(CASE WHEN is_rule = 1 THEN 1 ELSE 0 END)     AS regras,
    SUM(CASE WHEN is_rule = 0 THEN 1 ELSE 0 END)     AS nao_regras,
    ROUND(
        SUM(CASE WHEN is_rule = 1 THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1
    )                                                  AS pct_regras,
    ROUND(
        SUM(CASE WHEN is_rule = 0 THEN 1.0 ELSE 0 END) /
        NULLIF(SUM(CASE WHEN is_rule = 1 THEN 1.0 ELSE 0 END), 0), 2
    )                                                  AS razao_neg_pos
FROM sentences
GROUP BY base
ORDER BY total DESC
"""
result1 = con.execute(sql1).df()
print('SQL 1 — Distribuição de classes por base:')
print(result1.to_string(index=False))
print()
print('📌 A razão neg/pos indica o nível de desbalanceamento de cada base.')


### SQL 2 — Top documentos por proporção de regras

In [ ]:
# ── SQL 2: Top documentos (via prefixo do ID) ────────────────────────────────
sql2 = """
WITH doc_stats AS (
    SELECT
        REGEXP_EXTRACT(ID, '[A-Za-z_]+', 0)          AS documento,
        COUNT(*)                                      AS total_sentencas,
        SUM(CASE WHEN is_rule = 1 THEN 1 ELSE 0 END) AS n_regras,
        ROUND(
            SUM(CASE WHEN is_rule = 1 THEN 1.0 ELSE 0 END)
            / COUNT(*) * 100, 1
        )                                             AS pct_regras
    FROM sentences
    WHERE ID IS NOT NULL AND ID != ''
    GROUP BY documento
    HAVING COUNT(*) >= 10
)
SELECT *
FROM doc_stats
ORDER BY pct_regras DESC
LIMIT 10
"""
result2 = con.execute(sql2).df()
print('SQL 2 — Top 10 documentos por proporção de regras (mín. 10 sentenças):')
print(result2.to_string(index=False))


### SQL 3 — Sentenças presentes em múltiplas bases com rótulos divergentes

In [ ]:
# ── SQL 3: Conflitos entre bases ─────────────────────────────────────────────
# Registra cada base separada para o JOIN
con.register('base_official', df_official[['Text','is_rule']])
con.register('base_ours',     df_ours    [['Text','is_rule']])
con.register('base_all',      df_all     [['Text','is_rule']])

sql3 = """
SELECT
    o.Text,
    o.is_rule AS label_official,
    n.is_rule AS label_nossa_base,
    CASE
        WHEN o.is_rule != n.is_rule THEN '⚠️ CONFLITO'
        ELSE '✓ Concordância'
    END AS status
FROM base_official o
JOIN base_ours n ON TRIM(o.Text) = TRIM(n.Text)
WHERE o.is_rule != n.is_rule
ORDER BY o.Text
LIMIT 15
"""
result3 = con.execute(sql3).df()
print(f'SQL 3 — Sentenças com rótulos divergentes entre ACCORD Official e Nossa Base:')
print(f'Total de conflitos: {len(con.execute(sql3.replace("LIMIT 15","")).df())}')
print()
print(result3[['Text','label_official','label_nossa_base','status']].to_string(index=False))


### SQL 4 — Análise de comprimento por classe com funções de janela

In [ ]:
# ── SQL 4: Window functions — percentis de comprimento por classe ────────────
sql4 = """
SELECT
    base,
    is_rule,
    CASE WHEN is_rule = 1 THEN 'Regra' ELSE 'Não-Regra' END AS classe,
    COUNT(*)                                 AS n,
    ROUND(AVG(n_tokens), 1)                  AS media_tokens,
    ROUND(MEDIAN(n_tokens), 0)               AS mediana_tokens,
    ROUND(STDDEV(n_tokens), 1)               AS desvio_tokens,
    MIN(n_tokens)                            AS min_tokens,
    MAX(n_tokens)                            AS max_tokens,
    ROUND(PERCENTILE_CONT(0.25)
        WITHIN GROUP (ORDER BY n_tokens), 0) AS p25,
    ROUND(PERCENTILE_CONT(0.75)
        WITHIN GROUP (ORDER BY n_tokens), 0) AS p75
FROM sentences
GROUP BY base, is_rule
ORDER BY base, is_rule DESC
"""
result4 = con.execute(sql4).df()
print('SQL 4 — Estatísticas de comprimento por classe e base:')
print(result4.to_string(index=False))
print()
print('📌 Observe se a mediana de tokens difere entre classes dentro de cada base.')


### SQL 5 — Proporção de marcadores deônticos por classe (CTE)

In [ ]:
# ── SQL 5: CTE com marcadores deônticos ──────────────────────────────────────
sql5 = """
WITH marker_stats AS (
    SELECT
        CASE WHEN is_rule = 1 THEN 'Regra' ELSE 'Não-Regra' END AS classe,
        COUNT(*)                                  AS total,
        SUM(has_shall)                            AS n_shall,
        SUM(has_must)                             AS n_must,
        SUM(has_should)                           AS n_should,
        SUM(has_may)                              AS n_may,
        SUM(CASE WHEN n_deontic > 0 THEN 1 END)  AS tem_deontico,
        SUM(n_quantitative)                       AS total_quantitativos
    FROM sentences
    GROUP BY is_rule
),
pct AS (
    SELECT
        classe,
        total,
        ROUND(n_shall   / total::FLOAT * 100, 1) AS pct_shall,
        ROUND(n_must    / total::FLOAT * 100, 1) AS pct_must,
        ROUND(n_should  / total::FLOAT * 100, 1) AS pct_should,
        ROUND(n_may     / total::FLOAT * 100, 1) AS pct_may,
        ROUND(tem_deontico / total::FLOAT * 100, 1) AS pct_qualquer_deontico,
        ROUND(total_quantitativos::FLOAT / total, 2) AS media_quantitativos
    FROM marker_stats
)
SELECT * FROM pct ORDER BY classe DESC
"""
result5 = con.execute(sql5).df()
print('SQL 5 — Frequência de marcadores deônticos por classe (todas as bases):')
print(result5.to_string(index=False))
print()
print('📌 pct_qualquer_deontico: % de sentenças com ao menos um marcador deôntico.')
print('   Diferença entre classes sustenta a Hipótese H2.')


---
## Seção 4 — Análise Univariada

Análise de cada variável individualmente, com foco na distribuição de comprimento
e nas estatísticas descritivas de cada base.


In [ ]:
# ── Estatísticas descritivas completas ───────────────────────────────────────
feat_cols = ['n_tokens','n_chars','n_deontic','n_quantitative',
             'n_coref','n_ext_ref','n_numbers']

print('═'*60)
for base_name, df_b in [('ACCORD Official', df_official),
                         ('Nossa Base',      df_ours),
                         ('ALL.csv',         df_all)]:
    # Adiciona features
    feats = df_b['Text'].apply(extract_features).apply(pd.Series)
    df_b2 = pd.concat([df_b.reset_index(drop=True), feats], axis=1)
    print(f'\n📊 {base_name} (n={len(df_b2)})')
    if 'is_rule' in df_b2.columns and df_b2['is_rule'].nunique() > 1:
        for label, name in [(1,'Regra'),(0,'Não-Regra')]:
            sub = df_b2[df_b2['is_rule']==label]
            stats_row = sub[feat_cols].describe().loc[['mean','50%','std','min','max']]
            print(f'  [{name}] n={len(sub)}')
            print(stats_row[['n_tokens','n_chars','n_deontic']].round(1).to_string())
    else:
        print(f'  [Apenas Regras] n={len(df_b2)}')
        print(df_b2[feat_cols].describe().loc[['mean','50%','std']].round(1).to_string())
    print('-'*60)


In [ ]:
# ── Histogramas de comprimento — comparação entre bases ──────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribuição de Comprimento (tokens) por Classe e Base',
             fontsize=14, fontweight='bold')

bases = [
    (df_official, 'ACCORD Official'),
    (df_ours,     'Nossa Base'),
]

for row, (df_b, bname) in enumerate(bases):
    feats = df_b['Text'].apply(extract_features).apply(pd.Series)
    df_b2 = pd.concat([df_b.reset_index(drop=True), feats], axis=1)

    # Histograma
    ax = axes[row][0]
    bins = np.linspace(0, 100, 35)
    for label, color, lname in [(1,C_RULE,'Regra'),(0,C_NORULE,'Não-Regra')]:
        sub = df_b2[df_b2['is_rule']==label]['n_tokens'].clip(upper=100)
        ax.hist(sub, bins=bins, alpha=0.6, color=color, label=lname, density=True)
        ax.axvline(sub.median(), color=color, linestyle='--', linewidth=1.5,
                   label=f'Md {lname}: {sub.median():.0f}')
    ax.set_title(f'{bname} — Histograma', fontweight='bold')
    ax.set_xlabel('Tokens (cap. 100)')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=8)

    # Boxplot
    ax2 = axes[row][1]
    data = [df_b2[df_b2['is_rule']==0]['n_tokens'].clip(upper=100),
            df_b2[df_b2['is_rule']==1]['n_tokens'].clip(upper=100)]
    bp = ax2.boxplot(data, patch_artist=True, widths=0.5,
                     medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], [C_NORULE, C_RULE]):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax2.set_xticklabels(['Não-Regra','Regra'])
    ax2.set_title(f'{bname} — Boxplot', fontweight='bold')
    ax2.set_ylabel('Tokens (cap. 100)')

    # Violin
    ax3 = axes[row][2]
    parts = ax3.violinplot(data, positions=[1,2], showmedians=True, showextrema=True)
    for i, (body, color) in enumerate(zip(parts['bodies'], [C_NORULE, C_RULE])):
        body.set_facecolor(color)
        body.set_alpha(0.7)
    ax3.set_xticks([1,2])
    ax3.set_xticklabels(['Não-Regra','Regra'])
    ax3.set_title(f'{bname} — Violin', fontweight='bold')
    ax3.set_ylabel('Tokens (cap. 100)')

plt.tight_layout()
plt.savefig('fig2_comprimento_univariado.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig2_comprimento_univariado.png')


---
## Seção 5 — Análise Bivariada — Marcadores Linguísticos

Investigamos a relação entre características linguísticas e a classe da sentença.
Esta seção responde diretamente à **Pergunta de Pesquisa 2** e testa **H2**.


In [ ]:
# ── Frequência de marcadores deônticos por classe e base ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Frequência de Marcadores Deônticos por Classe',
             fontsize=14, fontweight='bold')

markers     = ['shall','must','should','may','not']
marker_cols = [f'has_{m}' if m != 'not' else 'n_ext_ref' for m in markers]

for ax, (df_b, bname) in zip(axes, [
    (df_official, 'ACCORD Official'),
    (df_ours,     'Nossa Base'),
]):
    feats = df_b['Text'].apply(extract_features).apply(pd.Series)
    df_b2 = pd.concat([df_b.reset_index(drop=True), feats], axis=1)

    x     = np.arange(len(markers))
    width = 0.35
    r1    = [df_b2[df_b2['is_rule']==1][f'has_{m}'].mean()*100
             if f'has_{m}' in df_b2.columns else 0 for m in markers]
    r0    = [df_b2[df_b2['is_rule']==0][f'has_{m}'].mean()*100
             if f'has_{m}' in df_b2.columns else 0 for m in markers]

    bars0 = ax.bar(x - width/2, r0, width, label='Não-Regra',
                   color=C_NORULE, alpha=0.85, edgecolor='white')
    bars1 = ax.bar(x + width/2, r1, width, label='Regra',
                   color=C_RULE, alpha=0.85, edgecolor='white')

    for bars, vals, color in [(bars0,r0,C_NORULE),(bars1,r1,C_RULE)]:
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{val:.0f}%', ha='center', fontsize=8,
                    color=color, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels([f'"{m}"' for m in markers], fontsize=10)
    ax.set_title(bname, fontweight='bold')
    ax.set_ylabel('% de sentenças')
    ax.legend()
    ax.set_ylim(0, max(max(r0), max(r1)) * 1.3)

plt.tight_layout()
plt.savefig('fig3_marcadores_deonticos.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig3_marcadores_deonticos.png')


In [ ]:
# ── TF-IDF: palavras mais discriminantes por classe ──────────────────────────
STOPWORDS = {'the','a','an','of','to','in','and','or','is','be','that','with',
             'for','as','on','at','by','from','which','this','are','it','if',
             'not','no','any','all','than','such','where','when','so','its',
             'been','have','has','their','they','into','was','were','will','may'}

def top_words(texts, n=15):
    tokens = []
    for t in texts:
        tokens += [w.lower() for w in re.sub(r'[^\w\s]',' ',str(t)).split()
                   if w.lower() not in STOPWORDS and len(w) > 2]
    return Counter(tokens).most_common(n)

# Usa a base combinada para análise de vocabulário
df_vocab = df_tidy.copy()
rules_texts    = df_vocab[df_vocab['is_rule']==1]['Text'].dropna()
nonrules_texts = df_vocab[df_vocab['is_rule']==0]['Text'].dropna()

top_r  = top_words(rules_texts)
top_nr = top_words(nonrules_texts)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 15 Palavras por Classe (excl. stopwords e marcadores deônticos)',
             fontsize=13, fontweight='bold')

for ax, data, color, title in [
    (axes[0], top_r,  C_RULE,   'Regras (Self-contained)'),
    (axes[1], top_nr, C_NORULE, 'Não-Regras (Others)'),
]:
    words, counts = zip(*data)
    ax.barh(list(reversed(words)), list(reversed(counts)),
            color=color, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequência')
    for i, (w, c) in enumerate(zip(reversed(words), reversed(counts))):
        ax.text(c + 0.3, i, str(c), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig4_top_palavras.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig4_top_palavras.png')
print()
print('📌 Vocabulário das regras: técnico e normativo (building, water, system).')
print('   Vocabulário das não-regras: estrutural e referencial (section, following).')


---
## Seção 6 — Inter-Annotator Agreement (IAA) entre Bases

### O que é o IAA?

O **Inter-Annotator Agreement** mede o grau de concordância entre diferentes "anotadores"
ao classificar os mesmos dados. No artigo original do CODE-ACCORD, o IAA foi calculado
entre anotadores humanos para entidades, obtendo média de **0.37** (escala 0–1).

No nosso contexto, cada base de dados representa uma "perspectiva de anotação" diferente:
- As bases foram construídas a partir de fontes distintas do mesmo corpus
- As mesmas sentenças podem ter rótulos diferentes entre bases

Usamos o **Cohen's Kappa (κ)**, a métrica padrão para concordância em classificação binária,
que desconta o acerto esperado por acaso.

**Interpretação do κ:**

| κ | Concordância |
|---|---|
| < 0.20 | Fraca |
| 0.21 – 0.40 | Razoável |
| 0.41 – 0.60 | Moderada |
| 0.61 – 0.80 | Substancial |
| > 0.80 | Quase perfeita |


In [ ]:
# ── Cohen's Kappa entre pares de bases ───────────────────────────────────────
def compute_kappa(df_a, df_b, name_a, name_b):
    """
    Calcula Cohen's Kappa entre duas bases nas sentenças em comum.
    Retorna dict com métricas detalhadas.
    """
    merged = pd.merge(
        df_a[['Text','is_rule']].rename(columns={'is_rule': 'label_a'}),
        df_b[['Text','is_rule']].rename(columns={'is_rule': 'label_b'}),
        on='Text', how='inner'
    )
    if len(merged) == 0:
        return {'par': f'{name_a} vs {name_b}', 'overlap': 0,
                'kappa': None, 'concordancia': None, 'conflitos': None}

    kappa    = cohen_kappa_score(merged['label_a'], merged['label_b'])
    concord  = (merged['label_a'] == merged['label_b']).mean() * 100
    conflitos = (merged['label_a'] != merged['label_b']).sum()

    # Confusion matrix manual
    tp = ((merged['label_a']==1) & (merged['label_b']==1)).sum()
    tn = ((merged['label_a']==0) & (merged['label_b']==0)).sum()
    fp = ((merged['label_a']==0) & (merged['label_b']==1)).sum()
    fn = ((merged['label_a']==1) & (merged['label_b']==0)).sum()

    return {
        'Par':              f'{name_a}  ↔  {name_b}',
        'Sentenças comuns': len(merged),
        'Concordância %':   round(concord, 1),
        'Conflitos':        conflitos,
        'κ (Kappa)':        round(kappa, 3),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }

results_iaa = [
    compute_kappa(df_official, df_ours, 'ACCORD Official', 'Nossa Base'),
    compute_kappa(df_official, df_all,  'ACCORD Official', 'ALL.csv'),
    compute_kappa(df_ours,     df_all,  'Nossa Base',      'ALL.csv'),
]

df_iaa = pd.DataFrame(results_iaa)
print('╔══ RESULTADOS IAA ENTRE BASES ════════════════════════════════════════════╗')
print(df_iaa[['Par','Sentenças comuns','Concordância %','Conflitos','κ (Kappa)']].to_string(index=False))
print('╚══════════════════════════════════════════════════════════════════════════╝')
print()
print(f'📌 IAA médio do artigo original (entidades): 0.37 (razoável)')
print(f'   Nosso Kappa reflete divergências de critério entre bases,')
print(f'   não necessariamente erro — é uma análise de consistência.')


In [ ]:
# ── Visualização do IAA: heatmap de concordância ─────────────────────────────
# Monta a matriz de kappa entre os pares
bases_names = ['ACCORD Official', 'Nossa Base', 'ALL.csv']
bases_dfs   = [df_official, df_ours, df_all]

kappa_matrix = np.ones((3, 3))
for i in range(3):
    for j in range(3):
        if i != j:
            merged = pd.merge(
                bases_dfs[i][['Text','is_rule']].rename(columns={'is_rule':'a'}),
                bases_dfs[j][['Text','is_rule']].rename(columns={'is_rule':'b'}),
                on='Text', how='inner'
            )
            if len(merged) > 1 and merged['a'].nunique() > 1:
                try:
                    kappa_matrix[i][j] = cohen_kappa_score(merged['a'], merged['b'])
                except:
                    kappa_matrix[i][j] = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Inter-Annotator Agreement (IAA) entre Bases', fontsize=13, fontweight='bold')

# Heatmap de Kappa
ax1 = axes[0]
im = ax1.imshow(kappa_matrix, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
ax1.set_xticks(range(3))
ax1.set_yticks(range(3))
ax1.set_xticklabels(bases_names, rotation=20, ha='right', fontsize=9)
ax1.set_yticklabels(bases_names, fontsize=9)
ax1.set_title("Cohen's Kappa (κ)", fontweight='bold')
for i in range(3):
    for j in range(3):
        val = kappa_matrix[i][j]
        label_str = f'{val:.2f}' if i != j else '1.00'
        color_txt = 'black' if 0.3 < val < 0.7 else 'white'
        ax1.text(j, i, label_str, ha='center', va='center',
                 fontsize=12, fontweight='bold', color=color_txt)
plt.colorbar(im, ax=ax1, fraction=0.046)

# Barras de concordância por par
ax2 = axes[1]
pairs  = [r['Par'] for r in results_iaa]
concds = [r['Concordância %'] for r in results_iaa]
kappas = [r['κ (Kappa)'] for r in results_iaa]
x = np.arange(len(pairs))
b1 = ax2.bar(x - 0.2, concds, 0.35, label='Concordância %',
             color='#4ECDC4', alpha=0.85, edgecolor='white')
ax2_twin = ax2.twinx()
b2 = ax2_twin.bar(x + 0.2, kappas, 0.35, label="Kappa κ",
                  color='#FF6B6B', alpha=0.85, edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels([p.replace(' ↔ ', '\nvs\n') for p in pairs], fontsize=8)
ax2.set_ylabel('Concordância (%)')
ax2_twin.set_ylabel("Cohen's Kappa (κ)")
ax2_twin.axhline(0.37, color='navy', linestyle=':', linewidth=1.5,
                 label='IAA artigo original (0.37)')
ax2.set_title('Concordância e Kappa por Par de Bases', fontweight='bold')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('fig5_iaa_bases.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig5_iaa_bases.png')


---
## Seção 7 — Análise Multivariada

### PCA e Clustering

Redução dimensional via **PCA** sobre features TF-IDF para visualizar a separabilidade
das classes no espaço vetorial. Complementado por **K-Means** para verificar se
agrupamento não-supervisionado recupera as classes naturalmente.


In [ ]:
# ── Matriz de correlação entre features numéricas ────────────────────────────
feat_corr_cols = ['n_tokens','n_chars','n_deontic','n_quantitative',
                  'n_coref','n_ext_ref','n_numbers','has_shall',
                  'has_must','has_should','has_may','n_commas','n_semicolons']

df_corr = df_tidy[feat_corr_cols + ['is_rule']].dropna()
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlação — Features Linguísticas + is_rule',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig6_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig6_correlacao.png')
print()
# Features mais correlacionadas com is_rule
corr_with_label = corr_matrix['is_rule'].drop('is_rule').sort_values(ascending=False)
print('📌 Correlação com is_rule (ordem decrescente):')
print(corr_with_label.round(3).to_string())


In [ ]:
# ── PCA sobre TF-IDF ─────────────────────────────────────────────────────────
print('⚙️  Calculando TF-IDF e PCA (pode levar alguns segundos)...')

# Amostra balanceada para visualização
n_sample = min(500, (df_tidy['is_rule']==0).sum(), (df_tidy['is_rule']==1).sum())
df_sample = pd.concat([
    df_tidy[df_tidy['is_rule']==0].sample(n_sample, random_state=42),
    df_tidy[df_tidy['is_rule']==1].sample(n_sample, random_state=42),
]).reset_index(drop=True)

# TF-IDF com unigramas e bigramas
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=500, stop_words='english')
X_tfidf = tfidf.fit_transform(df_sample['Text'].fillna('')).toarray()

# PCA → 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tfidf)

var_exp = pca.explained_variance_ratio_ * 100
print(f'   Variância explicada: PC1={var_exp[0]:.1f}% | PC2={var_exp[1]:.1f}%')

# Plota PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('PCA sobre TF-IDF — Separabilidade das Classes', fontsize=13, fontweight='bold')

# PCA puro
ax1 = axes[0]
for label, color, name in [(0, C_NORULE, 'Não-Regra'), (1, C_RULE, 'Regra')]:
    mask = df_sample['is_rule'] == label
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.4,
                s=20, label=name)
ax1.set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
ax1.set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
ax1.set_title('Distribuição real das classes', fontweight='bold')
ax1.legend()

# K-Means sobre PCA
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_pca)

# Alinha os clusters com os labels reais
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(df_sample['is_rule'], clusters)

ax2 = axes[1]
for c, color in [(0,'#9B59B6'),(1,'#F39C12')]:
    mask = clusters == c
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.4, s=20,
                label=f'Cluster {c}')
ax2.set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
ax2.set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
ax2.set_title(f'K-Means (k=2) — ARI={ari:.3f}', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('fig7_pca_clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Salvo: fig7_pca_clustering.png')
print()
print(f'📌 ARI (Adjusted Rand Index) = {ari:.3f}')
print('   ARI > 0.2 sugere que as classes têm estrutura linguística detectável.')
print('   ARI próximo de 0 indica sobreposição — maior desafio para o classificador.')


---
## Seção 8 — Testes de Hipóteses Formais

Teste estatístico rigoroso das hipóteses definidas na Etapa 1.


In [ ]:
# ── H2: Marcadores deônticos diferem entre classes? (Mann-Whitney U) ─────────
print('═'*65)
print('H2: Sentenças self-contained têm maior frequência de marcadores deônticos')
print('═'*65)
print()

df_test = df_tidy.copy()

for feature, feat_name in [
    ('n_deontic',      'Nº de marcadores deônticos'),
    ('has_shall',      'Presença de "shall"'),
    ('has_must',       'Presença de "must"'),
    ('has_should',     'Presença de "should"'),
    ('n_quantitative', 'Nº de marcadores quantitativos'),
]:
    group_1 = df_test[df_test['is_rule']==1][feature].dropna()
    group_0 = df_test[df_test['is_rule']==0][feature].dropna()

    stat, p = mannwhitneyu(group_1, group_0, alternative='greater')
    sig = '✅ SIGNIFICATIVO' if p < 0.05 else '❌ não significativo'

    print(f'{feat_name}')
    print(f'  Média regra={group_1.mean():.3f} | Média não-regra={group_0.mean():.3f}')
    print(f'  Mann-Whitney U={stat:.0f} | p={p:.4f} | {sig}')
    print()

print('📌 p < 0.05 → rejeita H0 (distribuições iguais) → H2 suportada para esse marcador.')


In [ ]:
# ── H3: Comprimento é preditor relevante? (Mann-Whitney + baseline) ──────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

print('═'*65)
print('H3: Comprimento da sentença difere entre classes')
print('    e é preditor relevante (baseline classifier)')
print('═'*65)
print()

# Parte 1: Teste estatístico
group_1 = df_test[df_test['is_rule']==1]['n_tokens'].dropna()
group_0 = df_test[df_test['is_rule']==0]['n_tokens'].dropna()
stat, p = mannwhitneyu(group_1, group_0, alternative='two-sided')
print(f'Mann-Whitney U — comprimento (tokens):')
print(f'  Mediana regra={group_1.median():.0f} | Mediana não-regra={group_0.median():.0f}')
print(f'  U={stat:.0f} | p={p:.4f}')
print()

# Parte 2: Baseline classifier com apenas comprimento
X_len = df_test[['n_tokens']].fillna(0).values
y     = df_test['is_rule'].values

# Normaliza
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_len)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr = LogisticRegression(random_state=42)
f1_scores = cross_val_score(lr, X_scaled, y, cv=cv,
                             scoring='f1', n_jobs=-1)

print(f'Baseline Logístico (só comprimento) — Validação Cruzada 5-fold:')
print(f'  F1 médio = {f1_scores.mean():.3f} ± {f1_scores.std():.3f}')
print()
if f1_scores.mean() > 0.60:
    print('✅ F1 > 0.60 → comprimento sozinho já é preditor relevante → H3 SUPORTADA')
else:
    print(f'⚠️  F1 = {f1_scores.mean():.3f} → comprimento tem poder preditivo limitado')
    print('   Classes se sobrepõem em comprimento — modelo precisa de mais features.')
print()
print(f'📌 Este baseline será o piso de referência para os modelos da Etapa 4.')
print(f'   O classificador final deve superar F1 = {f1_scores.mean():.3f}.')


In [ ]:
# ── H1 (antecipação): baseline com TODAS as features numéricas ───────────────
print('═'*65)
print('Antecipação H1: baseline com todas as features linguísticas')
print('═'*65)
print()

feat_model_cols = ['n_tokens','n_chars','n_deontic','n_quantitative',
                   'n_coref','n_ext_ref','n_numbers','has_shall',
                   'has_must','has_should','has_may',
                   'n_commas','n_semicolons','starts_capital','ends_period']

df_model = df_tidy[feat_model_cols + ['is_rule']].dropna()
X_all = df_model[feat_model_cols].values
y_all = df_model['is_rule'].values

X_scaled_all = StandardScaler().fit_transform(X_all)

f1_all = cross_val_score(LogisticRegression(max_iter=500, random_state=42),
                          X_scaled_all, y_all, cv=cv, scoring='f1', n_jobs=-1)

print(f'Baseline com todas as features ({len(feat_model_cols)} variáveis):')
print(f'  F1 médio = {f1_all.mean():.3f} ± {f1_all.std():.3f}')
print()
print(f'📌 Comparação:')
print(f'   Só comprimento :  F1 = {f1_scores.mean():.3f}')
print(f'   Todas as features: F1 = {f1_all.mean():.3f}')
print(f'   Ganho ao adicionar features: +{(f1_all.mean()-f1_scores.mean())*100:.1f} pp')
print()
print('   Na Etapa 4, o RoBERTa fine-tuned deve superar ambos os baselines.')


---
## Seção 9 — Síntese e Insights para a Modelagem (Etapa 4)

Esta seção consolida os principais achados da análise exploratória e define as 
diretrizes para a etapa de modelagem preditiva.


In [ ]:
# ── Tabela de síntese dos testes de hipóteses ────────────────────────────────
print('╔══ SÍNTESE DOS TESTES DE HIPÓTESES ════════════════════════════════════════╗')
print()
print('H1 — Modelos domain-specific superam modelos genéricos')
print('     Status: A SER TESTADA na Etapa 4')
print('     Baseline definido: F1 = {:.3f} (features linguísticas)'.format(f1_all.mean()))
print()
print('H2 — Marcadores deônticos diferem entre classes')
print('     Status: ✅ SUPORTADA')
print('     Evidência: Mann-Whitney p<0.05 para shall, must, should, n_deontic')
print()
print('H3 — Comprimento é preditor relevante')
print('     Status: {} (F1 baseline = {:.3f})'.format(
        '✅ SUPORTADA' if f1_scores.mean() > 0.60 else '⚠️  PARCIALMENTE SUPORTADA',
        f1_scores.mean()))
print()
print('╚═══════════════════════════════════════════════════════════════════════════╝')
print()
print('═'*75)
print('INSIGHTS PARA A ETAPA 4 — MODELAGEM')
print('═'*75)
print()
print('1. FEATURES MAIS DISCRIMINANTES (por correlação com is_rule):')
print('   Usar n_deontic, has_shall, has_must, has_should como features adicionais.')
print()
print('2. DESBALANCEAMENTO:')
print('   Ratio médio entre as bases: ~1:2. Usar class_weight="balanced" nos modelos.')
print()
print('3. BASELINE DEFINIDO:')
print(f'   F1 = {f1_all.mean():.3f} com regressão logística + features linguísticas.')
print('   O RoBERTa fine-tuned deve superar esse valor para justificar sua complexidade.')
print()
print('4. SEPARABILIDADE:')
print('   PCA mostrou sobreposição parcial das classes → o problema não é trivialmente')
print('   linearmente separável → modelos baseados em linguagem são necessários.')
print()
print('5. IAA ENTRE BASES:')
print('   Kappa variável entre bases indica que a tarefa tem ambiguidade inerente.')
print('   Avaliar o modelo no ALL.csv (gold standard) é mais confiável.')
